# Climate Indices with Earthkit & Xclim

This notebook demonstrates how to compute and visualize **climate indices** from CMIP6 datasets using the `earthkit-climate` and `xclim` packages.

We'll use:
- **Precipitation-based indices**:
  - *SDII*: Simple Daily Intensity Index (average precipitation on wet days)
  - *CWD*: Consecutive Wet Days (max number of wet days in a row)

- **Temperature-based indices**:
  - *DTR*: Daily Temperature Range (Tmax - Tmin)
  - *WSDI*: Warm Spell Duration Index (≥6 consecutive days above 90th percentile)
  - *HDD*: Heating Degree Days (based on temperature below threshold)

We’ll load **ACCESS-CM2 CMIP6 data** for both *historical* and *SSP585* scenarios.


In [ ]:
import earthkit.data as ekd
import matplotlib.pyplot as plt
import earthkit.plots as ekp
import cartopy.crs as ccrs

from earthkit.climate.indicators.precipitation import (
    daily_precipitation_intensity,
    maximum_consecutive_wet_days,
)
from earthkit.climate.indicators.temperature import (
    daily_temperature_range,
    warm_spell_duration_index,
    heating_degree_days,
)

plt.rcParams["figure.figsize"] = (8, 5)


## Loading CMIP6 data

We’ll use *daily gridded data* from the ACCESS-CM2 model for precipitation (`pr`), maximum (`tasmax`) and minimum (`tasmin`) temperature, for both historical and SSP585 future scenarios.


In [ ]:
# Load precipitation
pr_hist = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/pr_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
)
pr_ssp585 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/pr_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
)

# Load temperature
tasmin_hist = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmin_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
)
tasmin_ssp585 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmin_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
)

tasmax_hist = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmax_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
)
tasmax_ssp585 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmax_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
)


## Inspect and visualize the raw data

Before computing indices, let’s plot a few example grids to see how the raw variables look.


In [ ]:
# Define domain (Spain north coast)
domain = [-4.5, -4, 42.5, 43.4]

# Create the figure with 2x2 maps
figure = ekp.Figure(
    crs=ccrs.NearsidePerspective(central_longitude=-5, central_latitude=43),
    rows=2,
    columns=3,
    size=(15,10)
)

# Define variables and their datasets
variables = {
    "tasmin": (tasmin_hist, tasmin_ssp585, "celsius"),
    "tasmax": (tasmax_hist, tasmax_ssp585, "celsius"),
    "pr": (pr_hist, pr_ssp585, "mm/day"),
}

# HISTORICAL (row 0)
for col, (var, (hist, ssp, units)) in enumerate(variables.items()):
    hist_clim = hist.to_xarray().mean("time")
    cmap = "winter_r" if var == "pr" else "autumn"
    style = ekp.styles.Style(colors=cmap, units=units)
    map_plot = figure.add_map(row=0, column=col, domain=domain)
    map_plot.quickplot(hist_clim, style=style)
    map_plot.coastlines()
    map_plot.gridlines()
    map_plot.legend(location="right")
    map_plot.title(f"{var} Climatology (Historical)")

# SSP585 (row 1)
for col, (var, (hist, ssp, units)) in enumerate(variables.items()):
    ssp_clim = ssp.to_xarray().mean("time")
    cmap = "winter_r" if var == "pr" else "autumn_r"
    style = ekp.styles.Style(colors=cmap, units=units)
    map_plot = figure.add_map(row=1, column=col, domain=domain)
    map_plot.quickplot(ssp_clim, style=style)
    map_plot.coastlines()
    map_plot.gridlines()
    map_plot.legend(location="right")
    map_plot.title(f"{var} Climatology (SSP585)")

# Final layout
figure.show()

## Precipitation-based indices

We'll compute:
- **SDII** – Simple Daily Intensity Index (average rain on wet days)
- **CWD** – Consecutive Wet Days (max length of a wet period)


In [ ]:
# SDII
sdii = daily_precipitation_intensity(pr_ssp585, wet_day_threshold=1.0, frequency="YS")

# CWD
cwd = maximum_consecutive_wet_days(pr_ssp585, wet_day_threshold=1.0)


## Inspecting the computed precipitation indices

Now that we’ve calculated the precipitation-based indices (**SDII** and **CWD**),
let’s take a closer look at their structure and metadata.

We'll explore:
1. The list of fields available in each index object (`.ls()`).
2. The associated provenance and processing metadata (`.metadata()`).
3. The attributes of the resulting `xarray.Dataset` (`.to_xarray().attrs`).

This helps confirm that the computations ran correctly and to understand what information Earthkit keeps about each field.

In [ ]:
# Inspect the SDII object (Simple Daily Intensity Index)
print("SDII fields:")
print(sdii.ls())

print("\nSDII metadata:")
print(sdii.metadata()[0])  # show first metadata entry

print("\nSDII xarray attributes:")
print(sdii.to_xarray().attrs)


### Notes on SDII metadata

- The **metadata** includes details such as the processing history, indicator name, variable units, and temporal frequency.
- `earthkit-climate` attaches provenance automatically through its integration with `xclim`, ensuring full traceability.
- You can also explore the `sdii.to_xarray()` object directly to see the data variables and dimensions.


In [ ]:
# Inspect the CWD object (Maximum Consecutive Wet Days)
print("CWD fields:")
print(cwd.ls())

print("\nCWD metadata:")
print(cwd.metadata()[0])

print("\nCWD xarray attributes:")
print(cwd.to_xarray().attrs)


### Notes on CWD metadata

- The **CWD** (Consecutive Wet Days) index records the longest sequence of wet days (above a threshold, typically 1 mm/day) per period.
- Similar to SDII, it retains detailed provenance, so you know exactly which dataset, variable, and indicator were used.
- These attributes are critical for reproducibility in climate data analysis.


In [ ]:
# Define domain (Spain north coast)
domain = [-4.5, -4.0, 42.5, 43.4]

# Create figure: 1 row (2 indices) × 2 scenarios (historical + ssp585)
figure = ekp.Figure(
    crs=ccrs.NearsidePerspective(central_longitude=-5, central_latitude=43),
    rows=1,
    columns=2,
    size=(12, 6)
)

# Define indices and corresponding datasets
indices = {
    "SDII": sdii,
    "CWD": cwd
}

# Define color maps for each index
cmaps = {
    "SDII": "winter_r",
    "CWD": "winter_r"
}

units = {
    "SDII": "mm/day",
    "CWD": "days"
}

# PLOT: Each index climatology
for col, (name, index_obj) in enumerate(indices.items()):
    ds = index_obj.to_xarray().mean("time")
    cmap = cmaps[name]

    style = ekp.styles.Style(colors=cmap, units=units[name])
    map_plot = figure.add_map(row=0, column=col, domain=domain)
    map_plot.quickplot(ds, style=style)
    map_plot.coastlines()
    map_plot.gridlines()
    map_plot.title(f"{name} Climatology (SSP585)")
    map_plot.legend(location="right")

# Final layout
figure.show()

## Temperature-based indices

Now we’ll compute:
- **DTR** – Daily Temperature Range
- **WSDI** – Warm Spell Duration Index (based on 90th percentile)
- **HDD** – Heating Degree Days


In [ ]:
# DTR
dtr = daily_temperature_range(tasmax_ssp585, tasmin_ssp585)

# WSDI (using historical baseline)
wsdi = warm_spell_duration_index(tasmax_ssp585, tasmax_hist)

# HDD (approximation)
tas = (tasmax_ssp585.to_xarray()["tasmax"] + tasmin_ssp585.to_xarray()["tasmin"]) / 2
tas.attrs["units"] = "degC"
tas = tas.to_dataset(name="tas")
hdd = heating_degree_days(tasmax_ssp585, tasmin_ssp585, tas)


## Inspecting the temperature-based indices

Now let's explore the three temperature indices we calculated:

1. **DTR (Daily Temperature Range)** — Difference between daily maximum and minimum temperatures.
2. **WSDI (Warm Spell Duration Index)** — Number of warm spells: consecutive periods (≥6 days) above the 90th percentile of the historical period.
3. **HDD (Heating Degree Days)** — Heating degree days, estimating heating energy demand based on temperatures below a threshold.

For each index, we’ll check:
- The available fields (`.ls()`).
- The metadata and provenance (`.metadata()`).
- The dataset attributes (`.to_xarray().attrs`).

This helps us confirm that the results and units are consistent and properly documented.


In [ ]:
# DTR (Daily Temperature Range)
print("DTR fields:")
print(dtr.ls())

print("\n DTR metadata:")
print(dtr.metadata()[0])

print("\n DTR xarray attributes:")
print(dtr.to_xarray().attrs)


### Notes on DTR

- Represents the **daily temperature range**, a key measure of local temperature variability.
- The metadata contains information about the input variables (`tasmax`, `tasmin`), their units, and the indicator method used.
- Always check that the output units (`degC`) are correct and consistent with the inputs.


In [ ]:
# WSDI (Warm Spell Duration Index)
print("WSDI fields:")
wsdi.ls()

print("\n WSDI metadata:")
print(wsdi.metadata()[0])

print("\n WSDI xarray attributes:")
print(wsdi.to_xarray().attrs)


In [ ]:
# HDD (Heating Degree Days)
print("HDD fields:")
print(hdd.ls())

print("\n HDD metadata:")
print(hdd.metadata()[0])

print("\n HDD xarray attributes:")
print(hdd.to_xarray().attrs)

### Notes on HDD

- The **Heating Degree Days** index estimates heating demand based on temperatures below a threshold (typically 18 °C).
- Metadata document the base temperature, frequency of accumulation, and calculation method (approximation).
- HDD is a valuable indicator for **energy and climate impact assessments**.


In [ ]:
# Define domain (Spain north coast)
domain = [-4.5, -4.0, 42.5, 43.4]

# Create figure: 1 row (2 indices) × 2 scenarios (historical + ssp585)
figure = ekp.Figure(
    crs=ccrs.NearsidePerspective(central_longitude=-5, central_latitude=43),
    rows=1,
    columns=3,
    size=(12, 6)
)

# Define indices and corresponding datasets
indices = {
    "DTR": dtr,
    "WSDI": wsdi,
    "HDD": hdd
}

# Define color maps for each index
cmaps = {
    "DTR": "autumn_r",
    "WSDI": "autumn_r",
    "HDD": "autumn_r"
}

units = {
    "DTR": "K",
    "WSDI": "days",
    "HDD": "K days"
}

# PLOT: Each index climatology
for col, (name, index_obj) in enumerate(indices.items()):
    ds = index_obj.to_xarray().mean("time")
    cmap = cmaps[name]

    style = ekp.styles.Style(colors=cmap, units=units[name])
    map_plot = figure.add_map(row=0, column=col, domain=domain)
    map_plot.quickplot(ds, style=style)
    map_plot.coastlines()
    map_plot.gridlines()
    map_plot.title(f"{name} Climatology (SSP585)")
    map_plot.legend(location="right")

figure.show()

After visualizing the spatial climatologies, let's now explore the **temporal evolution** of each temperature-based index over the north coast of Spain.

We compute the **spatial mean** (averaging over latitude and longitude) and visualize how each index changes through time in the **SSP585 scenario**.

These time series help identify **long-term trends**, such as:
- Increasing warm spells (WSDI) under future scenarios.
- Decreasing heating requirements (HDD) in a warming climate.
- Changes in temperature variability (DTR).

In [ ]:
# Define units for each index
units = {
    "DTR": "celsius",
    "WSDI": "days",
    "HDD": "days",
}

# Plot spatial mean time series for each temperature-based index
for name, index_obj in {"DTR": dtr, "WSDI": wsdi, "HDD": hdd}.items():
    ds = index_obj.to_xarray()

    # Compute spatial mean over lat/lon
    ts = ds.mean(dim=["lat", "lon"])

    # Plot time series
    ekp.timeseries(
        ts,
        color="darkred",
        title=f"{name} — Spatial Mean over Northern Spain (SSP585)",
        xticks={
            "frequency": "Y",  # yearly ticks
            "format": "%Y",
        },
        units=units[name],
        labels="minor"
    ).show()